# EDA de viabilidad — Study Copilot

**Equipo:** Padilla, Marco; Sanches, Federico · **Fecha:** 11/10/2026

Produce los números del documento `docs/hito_2/2-eda-de-viabilidad.md`.
Corre de punta a punta sobre `datos/`, sin pasos manuales.
Una celda por propiedad y una por búsqueda de *leakage*.

Requiere `pypdf`, `pandas` y `matplotlib`.

## Celda 1 — Carga de la muestra

In [ ]:
import pathlib, re, hashlib, itertools, logging
import pypdf
import pandas as pd

RAIZ = pathlib.Path("datos")
HOY = 2026

def leer(p):
    if p.suffix.lower() == ".pdf":
        r = pypdf.PdfReader(str(p))
        return len(r.pages), "".join((pg.extract_text() or "") for pg in r.pages), (r.metadata or {})
    return None, p.read_text(encoding="utf-8", errors="replace"), {}

logging.getLogger("pypdf").setLevel(logging.ERROR)
filas, TEXTOS = [], {}
for p in sorted(RAIZ.rglob("*")):
    if not p.is_file():
        continue
    carrera, materia, tipo = p.parts[1], p.parts[2], p.parts[3]
    paginas, texto, md = leer(p)
    rel = p.relative_to(RAIZ).as_posix()
    TEXTOS[rel] = texto
    filas.append({
        "archivo": rel, "carrera": carrera, "materia": materia, "tipo": tipo,
        "ext": p.suffix.lower(), "kb": round(p.stat().st_size / 1024, 1),
        "paginas": paginas, "chars": len(texto.strip()),
        "sha1": hashlib.sha1(p.read_bytes()).hexdigest(),
        "autor_meta": str(md.get("/Author", "")).strip(),
        "fecha_meta": str(md.get("/CreationDate", ""))[2:10],
        "productor": str(md.get("/Producer", "")).strip(),
    })

inv = pd.DataFrame(filas)
print(f"archivos leidos: {len(inv)}")
inv[["archivo", "tipo", "ext", "paginas", "chars"]]

## Celda 2 — Volumen

In [ ]:
print(f"archivos            : {len(inv)}")
print(f"carreras            : {inv.carrera.nunique()} -> {sorted(inv.carrera.unique())}")
print(f"materias            : {inv.materia.nunique()} -> {sorted(inv.materia.unique())}")
print(f"paginas PDF         : {int(inv.paginas.fillna(0).sum())}")
print(f"caracteres de texto : {inv.chars.sum():,}")
print()
print(inv.pivot_table(index="materia", columns="tipo", values="archivo",
                      aggfunc="count", fill_value=0).to_string())
print()
print("NIVEL EXIGIDO (caso B, material complementario): >=50 piezas revisadas")
print(f"MUESTRA: {len(inv)} piezas -> NO CUMPLE")
print("NIVEL EXIGIDO (caso A, planes): >=30 planes sobre >=3 materias")
print("MUESTRA: 0 planes -> NO CUMPLE (la muestra no contiene datos de uso)")

## Celda 3 — Completitud

In [ ]:
mat = sorted(inv.materia.unique())
con_temario = sorted(inv[inv.tipo == "temario"].materia.unique())
sin_temario = sorted(set(mat) - set(con_temario))

print(f"materias con temario       : {len(con_temario)}/{len(mat)} -> {con_temario}")
print(f"materias SIN temario       : {sin_temario}")
print(f"archivos con fecha en meta : {(inv.fecha_meta.str.len() == 8).sum()}/{len(inv)}")
print(f"archivos con autor en meta : {(inv.autor_meta != '').sum()}/{len(inv)}")
print(f"archivos con texto vacio   : {(inv.chars == 0).sum()}/{len(inv)}")
print()
print("El temario es lo que permite saber que temas entran y cuanto pesan.")
print(f"Sin el, {len(sin_temario)} de {len(mat)} materias no pueden usarse para priorizar.")

## Celda 4 — Corrección

In [ ]:
print(f"duplicados exactos (SHA-1)    : {int(inv.duplicated('sha1').sum())}")
print(f"formatos                      : {dict(inv.ext.value_counts())}")
npdf = int((inv.ext == '.pdf').sum())
print(f"PDFs sin capa de texto (OCR)  : {int(((inv.ext == '.pdf') & (inv.chars == 0)).sum())}/{npdf}")

class Cap(logging.Handler):
    def __init__(self):
        super().__init__(); self.msgs = []
    def emit(self, r):
        self.msgs.append(r.getMessage())

log = logging.getLogger("pypdf"); log.setLevel(logging.WARNING)
malos = []
for p in sorted(RAIZ.rglob("*.pdf")):
    h = Cap(); log.addHandler(h)
    try:
        for pg in pypdf.PdfReader(str(p)).pages:
            pg.extract_text()
    except Exception as e:
        h.msgs.append(f"EXC {e}")
    log.removeHandler(h)
    if h.msgs:
        malos.append((p.relative_to(RAIZ).as_posix(), h.msgs[0][:70]))
log.setLevel(logging.ERROR)

print(f"PDFs con estructura malformada: {len(malos)}/{npdf}")
for a, m in malos:
    print(f"   {a} -> {m}")

## Celda 5 — Vigencia

In [ ]:
RE_ANIO = re.compile("(?<![0-9])(19[89][0-9]|20[0-2][0-9])(?![0-9])")

def anios_texto(t):
    return sorted({int(y) for y in RE_ANIO.findall(t) if 1989 < int(y) <= HOY})

filas = []
for _, r in inv.iterrows():
    a_meta = int(r.fecha_meta[:4]) if len(r.fecha_meta) == 8 else None
    a_txt = anios_texto(TEXTOS[r.archivo])
    cand = [a for a in ([a_meta] + a_txt[-1:]) if a]
    ref = max(cand) if cand else None
    filas.append({"archivo": r.archivo, "anio_meta": a_meta,
                  "anios_texto": str(a_txt[-4:]), "anio_ref": ref})

vig = pd.DataFrame(filas)
vig["anio_ref"] = pd.to_numeric(vig["anio_ref"], errors="coerce")
vig["antiguedad"] = HOY - vig["anio_ref"]
print(vig.to_string(index=False))

ref = vig.anio_ref.dropna()
print(f"\nsin ninguna fecha determinable : {int(vig.anio_ref.isna().sum())}/{len(vig)}")
print(f"rango de fechas                : {int(ref.min())} - {int(ref.max())}")
print(f"mas reciente de la muestra     : {int(ref.max())} ({HOY - int(ref.max())} anios de antiguedad)")

es_tem = vig.archivo.str.contains("temario")
print(f"\nNIVEL temario <=1 anio             : cumplen {int((vig[es_tem].antiguedad <= 1).sum())}/{int(es_tem.sum())}")
ex = vig[~es_tem]
print(f"NIVEL examenes/practicas <=3 anios : cumplen {int((ex.antiguedad <= 3).sum())}/{len(ex)}")

## Celda 6 — Representatividad

In [ ]:
import matplotlib.pyplot as plt

# "fuera del area del equipo" se evalua a mano: las 3 materias son de Ing. en Sistemas.
FUERA_DEL_AREA = 0
NIVEL = {"carreras": 2, "materias": 3, "materias fuera del area del equipo": 1}
REAL = {"carreras": inv.carrera.nunique(), "materias": inv.materia.nunique(),
        "materias fuera del area del equipo": FUERA_DEL_AREA}

for k in NIVEL:
    print(f"{k:36s} exigido {NIVEL[k]} | muestra {REAL[k]} | "
          f"{'CUMPLE' if REAL[k] >= NIVEL[k] else 'NO CUMPLE'}")
print(f"\ninstituciones distintas: 1 (UTN FR Delta en todos los encabezados)")

fig, ax = plt.subplots(figsize=(7, 2.8))
inv.groupby(["materia", "tipo"]).size().unstack(fill_value=0).plot(kind="barh", stacked=True, ax=ax)
ax.set_xlabel("archivos"); ax.set_ylabel("")
ax.set_title(f"Composicion de la muestra por materia y tipo (n={len(inv)})")
plt.tight_layout(); plt.show()

## Celda 7 — Confiabilidad

In [ ]:
MARCAS = ["se deja a cargo de los alumnos", "posibles preguntas", "estan incompletas",
          "están incompletas", "estas fueron las preguntas", "se pide que se modifique"]

filas = []
for a, t in TEXTOS.items():
    # el texto extraido corta las frases con saltos de linea: hay que normalizar antes de buscar
    low = " ".join(t.lower().split())
    hits = [m for m in MARCAS if m in low]
    if hits:
        filas.append({"archivo": a, "marca_hallada": hits[0], "chars": len(t.strip())})

est = pd.DataFrame(filas)
print(est.to_string(index=False))
tot = int(inv.chars.sum())
print(f"\narchivos de origen estudiantil declarado: {len(est)}/{len(inv)}")
print(f"volumen que concentran: {int(est.chars.sum()):,} de {tot:,} chars "
      f"({100 * est.chars.sum() / tot:.0f}% del texto de la muestra)")
print("revisores que etiquetaron la muestra: 0 -> acuerdo entre revisores no calculable")
print("\nnombres de persona expuestos en metadatos:")
print(inv[inv.autor_meta != ""][["archivo", "autor_meta"]].to_string(index=False))

## Celda 8 — *Leakage* 1: la definición

Columnas que se completan después del momento en que el producto tendría que predecir.

In [ ]:
print(f"archivos de la muestra que son registro de uso: 0/{len(inv)}")
print("La muestra es material de catedra, no registros de uso del producto.")
print("No contiene la etiqueta del producto (el feedback de si el plan sirvio),")
print("por lo tanto no hay columna posterior a la prediccion que pueda filtrarse.")
print("\nNo encontramos leakage por definicion en esta muestra.")
print("Metodo: la muestra se inspecciono archivo por archivo (celda 1); ninguno")
print("contiene campos generados despues de una prediccion.")
print("\nATENCION: el riesgo sigue vigente para el dataset futuro, no para esta muestra.")

## Celda 9 — *Leakage* 2: el tiempo

In [ ]:
print(f"archivos con fecha determinable: {int(vig.anio_ref.notna().sum())}/{len(vig)}")
print(f"archivos SIN fecha determinable: {int(vig.anio_ref.isna().sum())}/{len(vig)}")
print(f"rango temporal de la muestra   : {int(ref.min())} - {int(ref.max())}")
print()
print("La muestra no tiene etiqueta, asi que no hay dependencia temporal etiqueta/fecha.")
print(f"Pero {int(vig.anio_ref.isna().sum())} archivos no se pueden ordenar en el tiempo:")
print("si mas adelante hiciera falta un corte temporal, esos quedan sin poder ubicarse.")

## Celda 10 — *Leakage* 3: los duplicados

In [ ]:
print(f"duplicados exactos (SHA-1): {int(inv.duplicated('sha1').sum())}")

TOK = re.compile("[a-záéíóúñ]{5,}")
sets = {a: set(TOK.findall(t.lower())) for a, t in TEXTOS.items()}
pares = []
for a, b in itertools.combinations(sets, 2):
    if not sets[a] or not sets[b]:
        continue
    j = len(sets[a] & sets[b]) / len(sets[a] | sets[b])
    if j > 0.30:
        pares.append((round(j, 2), a, b))

print(f"casi-duplicados (Jaccard > 0.30): {len(pares)}")
for j, a, b in sorted(pares, reverse=True):
    print(f"   {j}  {a}  <->  {b}")

## Celda 11 — *Leakage* 4: los grupos

In [ ]:
print(inv.groupby("materia").size().to_string())
print(f"\nunidad de agrupamiento = materia ({inv.materia.nunique()} grupos)")
print("Varios archivos pertenecen a la misma catedra y comparten enunciados y")
print("vocabulario. Partir por archivo pondria la misma catedra en train y eval.")
print("La particion tiene que ser por materia.")
print(f"\nCon {inv.materia.nunique()} grupos, cualquier particion deja 1 o 2 materias de un lado:")
print("la evaluacion seria sobre una sola catedra y no es generalizable.")

## Celda 12 — *Leakage* 5: la generación

In [ ]:
print(inv[["archivo", "productor", "autor_meta"]].to_string(index=False))
print()
print(f"productores distintos             : {inv[inv.productor != ''].productor.nunique()}")
print(f"archivos sin metadato de productor: {int((inv.productor == '').sum())}/{len(inv)}")
print()
print("A quien se le pregunto como se genero el dato: a nadie.")
print("Los archivos se descargaron de repositorios estudiantiles; no se contacto")
print("a ninguna catedra para confirmar que sean las versiones vigentes ni oficiales.")